# 2층 신경망 역전파 유도와 손계산

입력 2개 → 은닉 2개 → 출력 1개인 신경망에서 손실 $L$을 각 가중치로 미분하는 과정을 연쇄 법칙(Chain Rule)으로 유도하고, 과제의 고정 예제로 손계산한 뒤 NumPy 코드(`src/backprop.py`)로 같은 값이 나오는지 확인한다.

| 장 | 내용 |
|---|---|
| 1 | 신경망 구조와 기호, shape |
| 2 | 순전파 수식 |
| 3 | 역전파 수식 유도 (연쇄 법칙) |
| 4 | 고정 예제 손계산 (순전파 → 역전파, 체크포인트) |
| 5 | NumPy 코드 검증 (소수점 4자리 비교) |
| 6 | 수치 미분으로 한 번 더 검증 (Gradient Check) |

## 1. 신경망 구조와 기호

```
x (2,) ──W1 (2,2), b1 (2,)──> z1 (2,) ──σ──> a1 (2,) ──W2 (2,), b2──> z2 ──σ──> ŷ ──BCE──> L
```

| 기호 | 뜻 | shape | 코드 이름 (`src/backprop.py`) |
|---|---|---|---|
| $x$ | 입력 | (2,) | `input_vector` |
| $W_1$ | 은닉층 가중치 (행: 은닉 뉴런, 열: 입력) | (2, 2) | `hidden_weights` |
| $b_1$ | 은닉층 편향 | (2,) | `hidden_bias` |
| $z_1$ | 은닉층 선형 결합 | (2,) | `hidden_pre_activation` |
| $a_1$ | 은닉층 활성화 출력 | (2,) | `hidden_activation` |
| $W_2$ | 출력층 가중치 | (2,) | `output_weights` |
| $b_2$ | 출력층 편향 | 스칼라 | `output_bias` |
| $z_2$ | 출력층 선형 결합 | 스칼라 | `output_pre_activation` |
| $\hat{y}$ | 예측 확률 (`y_pred`) | 스칼라 | `prediction` |
| $L$ | Binary Cross-Entropy 손실 | 스칼라 | `loss` |

- 활성화 함수는 Sigmoid $\sigma(z) = \dfrac{1}{1 + e^{-z}}$ 이다.
- Sigmoid의 미분은 $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$ 이다. 출력값만 알면 미분값을 바로 구할 수 있다.

## 2. 순전파 수식

$$
\begin{aligned}
z_1 &= W_1 x + b_1 & &(2,2)\,@\,(2,) + (2,) \rightarrow (2,) \\
a_1 &= \sigma(z_1) & &(2,) \text{ 원소별} \\
z_2 &= W_2 \cdot a_1 + b_2 & &(2,) \cdot (2,) + \text{스칼라} \rightarrow \text{스칼라} \\
\hat{y} &= \sigma(z_2) & &\text{스칼라} \\
L &= -\big(y \log \hat{y} + (1 - y) \log (1 - \hat{y})\big) & &\text{스칼라}
\end{aligned}
$$

## 3. 역전파 수식 유도 (출력층 → 입력층)

$L$은 $W_1$을 직접 알지 못한다. $W_1 \rightarrow z_1 \rightarrow a_1 \rightarrow z_2 \rightarrow \hat{y} \rightarrow L$ 의 사슬로만 이어져 있다. 그래서 각 고리의 국소 미분을 곱해 나가는 **연쇄 법칙**이 필요하다. 출력 쪽부터 구한 값을 다음 단계가 그대로 재사용하므로 계산이 한 번씩만 일어난다.

### 3.1 $\partial L / \partial \hat{y}$ — shape: 스칼라

$$
\frac{\partial L}{\partial \hat{y}} = -\frac{y}{\hat{y}} + \frac{1 - y}{1 - \hat{y}}
$$

### 3.2 $\partial L / \partial z_2$ — shape: 스칼라 (체크포인트)

$\hat{y} = \sigma(z_2)$ 이므로 $\partial \hat{y} / \partial z_2 = \hat{y}(1 - \hat{y})$ 이다.

$$
\frac{\partial L}{\partial z_2}
= \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_2}
= \left(-\frac{y}{\hat{y}} + \frac{1 - y}{1 - \hat{y}}\right)\hat{y}(1 - \hat{y})
= -y(1 - \hat{y}) + (1 - y)\hat{y}
= \hat{y} - y
$$

- Sigmoid와 BCE를 함께 쓰면 분모가 지워져 "예측 - 정답"이라는 간단한 식이 된다.

### 3.3 $\partial L / \partial W_2$, $\partial L / \partial b_2$ — shape: (2,), 스칼라

$z_2 = W_{2,1} a_{1,1} + W_{2,2} a_{1,2} + b_2$ 이므로 $\partial z_2 / \partial W_{2,j} = a_{1,j}$, $\partial z_2 / \partial b_2 = 1$ 이다.

$$
\frac{\partial L}{\partial W_2} = \frac{\partial L}{\partial z_2}\, a_1 \quad (\text{스칼라} \times (2,) \rightarrow (2,)), \qquad
\frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial z_2}
$$

### 3.4 $\partial L / \partial a_1$ — shape: (2,)

$\partial z_2 / \partial a_{1,j} = W_{2,j}$ 이다.

$$
\frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2}\, W_2 \quad (\text{스칼라} \times (2,) \rightarrow (2,))
$$

### 3.5 $\partial L / \partial z_1$ — shape: (2,) (체크포인트)

$a_{1,j} = \sigma(z_{1,j})$ 는 원소별 함수라서 $\partial a_{1,j} / \partial z_{1,j} = a_{1,j}(1 - a_{1,j})$ 이고, 다른 원소끼리는 영향이 없다. 그래서 원소별 곱($\odot$)이 된다.

$$
\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \odot \sigma'(z_1) = \frac{\partial L}{\partial a_1} \odot a_1 (1 - a_1) \quad ((2,) \odot (2,) \rightarrow (2,))
$$

### 3.6 $\partial L / \partial W_1$, $\partial L / \partial b_1$ — shape: (2, 2), (2,)

$z_{1,i} = \sum_j W_{1,ij} x_j + b_{1,i}$ 이므로 $\partial z_{1,i} / \partial W_{1,ij} = x_j$ 이다. $i$행 $j$열 원소가 $\dfrac{\partial L}{\partial z_{1,i}} x_j$ 이므로 외적(outer product)이다.

$$
\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial z_1} \otimes x = \operatorname{outer}\!\left(\frac{\partial L}{\partial z_1}, x\right) \quad ((2,) \otimes (2,) \rightarrow (2,2)), \qquad
\frac{\partial L}{\partial b_1} = \frac{\partial L}{\partial z_1}
$$

- 기울기의 shape은 항상 그 파라미터의 shape과 같다. ($W_1$: (2,2) → $\partial L/\partial W_1$: (2,2)) 그래야 $W_1 \leftarrow W_1 - \eta\,\partial L/\partial W_1$ 로 갱신할 수 있다.

## 4. 고정 예제 손계산

과제가 정한 값으로 계산한다. 과제 문서에서 `y_true`가 비어 있어 결과 예시와 같은 $y = 1$ 을 쓴다.

| 항목 | 값 | shape |
|---|---|---|
| $x$ | [1, 0] | (2,) |
| $W_1$ | [[0.1, 0.2], [0.3, 0.4]] | (2, 2) |
| $b_1$ | [0, 0] | (2,) |
| $W_2$ | [0.5, 0.6] | (2,) |
| $b_2$ | 0 | 스칼라 |
| $y$ | 1 | 스칼라 |

반올림 규칙
- 중간값은 소수점 6자리까지 들고 가고, 마지막 결과만 4자리로 반올림한다.
- 중간값을 먼저 4자리로 반올림하면 $z_2$가 0.6071로 나온다. ($0.5 \times 0.5250 + 0.6 \times 0.5744 = 0.60714$) 6자리로 계산하면 0.607155 → **0.6072** 이고, NumPy 결과와도 이 값이 맞는다.

### 4.1 순전파 손계산

| 단계 | 계산 | 결과 (6자리) | 4자리 | shape |
|---|---|---|---|---|
| $z_1 = W_1 x + b_1$ | $[0.1 \cdot 1 + 0.2 \cdot 0,\; 0.3 \cdot 1 + 0.4 \cdot 0] + [0, 0]$ | [0.100000, 0.300000] | [0.1000, 0.3000] | (2,) |
| $a_1 = \sigma(z_1)$ | $1/(1 + e^{-0.1}) = 1/1.904837$, $\;1/(1 + e^{-0.3}) = 1/1.740818$ | [0.524979, 0.574443] | [0.5250, 0.5744] | (2,) |
| $z_2 = W_2 \cdot a_1 + b_2$ | $0.5 \times 0.524979 + 0.6 \times 0.574443 + 0 = 0.262490 + 0.344666$ | 0.607155 | 0.6072 | 스칼라 |
| $\hat{y} = \sigma(z_2)$ | $1 / (1 + e^{-0.607155}) = 1 / 1.544900$ | 0.647292 | 0.6473 | 스칼라 |
| $L$ | $-(1 \cdot \log 0.647292 + 0 \cdot \log 0.352708)$ | 0.434958 | 0.4350 | 스칼라 |

### 4.2 역전파 손계산

| 단계 | 계산 | 결과 (6자리) | 4자리 | shape |
|---|---|---|---|---|
| $\partial L/\partial \hat{y}$ | $-1/0.647292 + 0$ | −1.544899 | −1.5449 | 스칼라 |
| $\sigma'(z_2)$ | $0.647292 \times (1 - 0.647292)$ | 0.228305 | 0.2283 | 스칼라 |
| **$\partial L/\partial z_2$** (체크포인트) | $-1.544899 \times 0.228305$ $\;(= \hat{y} - y = 0.647292 - 1)$ | −0.352708 | **−0.3527** | 스칼라 |
| $\partial L/\partial W_2$ | $-0.352708 \times [0.524979, 0.574443]$ | [−0.185165, −0.202611] | [−0.1852, −0.2026] | (2,) |
| $\partial L/\partial b_2$ | $= \partial L/\partial z_2$ | −0.352708 | −0.3527 | 스칼라 |
| $\partial L/\partial a_1$ | $-0.352708 \times [0.5, 0.6]$ | [−0.176354, −0.211625] | [−0.1764, −0.2116] | (2,) |
| $\sigma'(z_1)$ | $[0.524979 \times 0.475021,\; 0.574443 \times 0.425557]$ | [0.249376, 0.244458] | [0.2494, 0.2445] | (2,) |
| **$\partial L/\partial z_1$** (체크포인트) | $[-0.176354 \times 0.249376,\; -0.211625 \times 0.244458]$ | [−0.043979, −0.051734] | **[−0.0440, −0.0517]** | (2,) |
| $\partial L/\partial W_1$ | $\operatorname{outer}([-0.043979, -0.051734], [1, 0])$ | [[−0.043979, 0], [−0.051734, 0]] | [[−0.0440, 0], [−0.0517, 0]] | (2, 2) |
| $\partial L/\partial b_1$ | $= \partial L/\partial z_1$ | [−0.043979, −0.051734] | [−0.0440, −0.0517] | (2,) |

- $\partial L/\partial W_1$의 둘째 열이 0인 이유: $x_2 = 0$ 이라 $W_{1,i2}$가 바뀌어도 $z_1$이 변하지 않는다.
- 과제 결과 예시의 $\partial L/\partial z_1$ 둘째 값(−0.0513)과 다르다. $\sigma'(0.3) = 0.244458$ 이므로 $-0.211625 \times 0.244458 = -0.051734$ 이고, 아래 NumPy 결과와 수치 미분 결과도 −0.0517 이다.

## 5. NumPy 코드 검증

`src/backprop.py`의 `forward_pass()`, `backward_pass()`로 같은 계산을 하고, 4장의 손계산 4자리 값과 비교한다.

In [1]:
# 경로 설정과 모듈 불러오기
import sys
from pathlib import Path

import numpy as np

# 노트북 폴더의 한 단계 위(프로젝트 최상위)를 import 경로에 넣는다
sys.path.insert(0, str(Path.cwd().parent))

# 역전파 모듈에서 필요한 함수를 가져온다
from src.backprop import (
    backward_pass,
    forward_pass,
    make_fixed_example,
    numerical_hidden_weight_gradient,
)

# 과제 고정 예제를 불러온다
example = make_fixed_example()

# 순전파를 실행한다
forward_result = forward_pass(example.input_vector, example.parameters, example.target)

# 역전파를 실행한다
gradients = backward_pass(forward_result, example.parameters, example.target)

In [2]:
# 순전파 중간값을 shape과 함께 출력한다
print("순전파 (shape 명시)")

# 이름, shape, 값 목록
forward_rows = [
    # 입력 x
    ("x", example.input_vector),
    # 은닉층 가중치 W1
    ("W1", example.parameters.hidden_weights),
    # z1 = W1 @ x + b1
    ("z1 = W1 @ x + b1", forward_result.hidden_pre_activation),
    # a1 = sigmoid(z1)
    ("a1 = sigmoid(z1)", forward_result.hidden_activation),
    # 출력층 가중치 W2
    ("W2", example.parameters.output_weights),
    # z2 = W2 @ a1 + b2
    ("z2 = W2 @ a1 + b2", np.float64(forward_result.output_pre_activation)),
    # y_pred = sigmoid(z2)
    ("y_pred = sigmoid(z2)", np.float64(forward_result.prediction)),
    # BCE 손실
    ("L (BCE)", np.float64(forward_result.loss)),
]

# 한 줄씩 출력한다
for row_name, row_value in forward_rows:
    # 스칼라면 "scalar", 배열이면 shape을 적는다
    shape_text = "scalar" if np.ndim(row_value) == 0 else str(np.shape(row_value))

    # 이름, shape, 4자리 값을 출력한다
    print(f"  {row_name:<22} {shape_text:<7} = {np.round(row_value, 4).tolist()}")

순전파 (shape 명시)
  x                      (2,)    = [1.0, 0.0]
  W1                     (2, 2)  = [[0.1, 0.2], [0.3, 0.4]]
  z1 = W1 @ x + b1       (2,)    = [0.1, 0.3]
  a1 = sigmoid(z1)       (2,)    = [0.525, 0.5744]
  W2                     (2,)    = [0.5, 0.6]
  z2 = W2 @ a1 + b2      scalar  = 0.6072
  y_pred = sigmoid(z2)   scalar  = 0.6473
  L (BCE)                scalar  = 0.435


In [3]:
# 역전파 결과를 shape과 함께 출력한다
print(f"역전파 (y_true={example.target:g}, BCE)")

# 이름, 값 목록
backward_rows = [
    # ∂L/∂y_pred
    ("dL/dy_pred", np.float64(gradients.prediction)),
    # ∂L/∂z2 (체크포인트)
    ("dL/dz2  <- checkpoint", np.float64(gradients.output_pre_activation)),
    # ∂L/∂W2
    ("dL/dW2", gradients.output_weights),
    # ∂L/∂b2
    ("dL/db2", np.float64(gradients.output_bias)),
    # ∂L/∂a1
    ("dL/da1", gradients.hidden_activation),
    # ∂L/∂z1 (체크포인트)
    ("dL/dz1  <- checkpoint", gradients.hidden_pre_activation),
    # ∂L/∂W1
    ("dL/dW1", gradients.hidden_weights),
    # ∂L/∂b1
    ("dL/db1", gradients.hidden_bias),
]

# 한 줄씩 출력한다
for row_name, row_value in backward_rows:
    # 스칼라면 "scalar", 배열이면 shape을 적는다
    shape_text = "scalar" if np.ndim(row_value) == 0 else str(np.shape(row_value))

    # -0.0을 0.0으로 정리해 4자리 값을 만든다
    rounded_value = np.round(row_value, 4) + 0.0

    # 이름, shape, 값을 출력한다
    print(f"  {row_name:<22} {shape_text:<7} = {rounded_value.tolist()}")

역전파 (y_true=1, BCE)
  dL/dy_pred             scalar  = -1.5449
  dL/dz2  <- checkpoint  scalar  = -0.3527
  dL/dW2                 (2,)    = [-0.1852, -0.2026]
  dL/db2                 scalar  = -0.3527
  dL/da1                 (2,)    = [-0.1764, -0.2116]
  dL/dz1  <- checkpoint  (2,)    = [-0.044, -0.0517]
  dL/dW1                 (2, 2)  = [[-0.044, 0.0], [-0.0517, 0.0]]
  dL/db1                 (2,)    = [-0.044, -0.0517]


In [4]:
# 4장 손계산 표의 4자리 값 (손으로 적은 값)
hand_calculated_values = {
    # 순전파 z1
    "z1": [0.1000, 0.3000],
    # 순전파 a1
    "a1": [0.5250, 0.5744],
    # 순전파 z2
    "z2": 0.6072,
    # 순전파 y_pred
    "y_pred": 0.6473,
    # 역전파 ∂L/∂y_pred
    "dL/dy_pred": -1.5449,
    # 역전파 ∂L/∂z2 (체크포인트)
    "dL/dz2": -0.3527,
    # 역전파 ∂L/∂W2
    "dL/dW2": [-0.1852, -0.2026],
    # 역전파 ∂L/∂a1
    "dL/da1": [-0.1764, -0.2116],
    # 역전파 ∂L/∂z1 (체크포인트)
    "dL/dz1": [-0.0440, -0.0517],
    # 역전파 ∂L/∂W1
    "dL/dW1": [[-0.0440, 0.0], [-0.0517, 0.0]],
}

# 같은 이름의 NumPy 계산값
numpy_values = {
    # 순전파 z1
    "z1": forward_result.hidden_pre_activation,
    # 순전파 a1
    "a1": forward_result.hidden_activation,
    # 순전파 z2
    "z2": forward_result.output_pre_activation,
    # 순전파 y_pred
    "y_pred": forward_result.prediction,
    # 역전파 ∂L/∂y_pred
    "dL/dy_pred": gradients.prediction,
    # 역전파 ∂L/∂z2
    "dL/dz2": gradients.output_pre_activation,
    # 역전파 ∂L/∂W2
    "dL/dW2": gradients.output_weights,
    # 역전파 ∂L/∂a1
    "dL/da1": gradients.hidden_activation,
    # 역전파 ∂L/∂z1
    "dL/dz1": gradients.hidden_pre_activation,
    # 역전파 ∂L/∂W1
    "dL/dW1": gradients.hidden_weights,
}

# 항목별 일치 여부를 모은다
match_results = []

# 표 머리글을 출력한다
print(f"{'항목':<11} {'손계산':<28} {'NumPy (4자리 반올림)':<28} 일치")

# 항목마다 비교한다
for value_name, hand_value in hand_calculated_values.items():
    # NumPy 값을 4자리로 반올림한다 (-0.0은 0.0으로)
    numpy_rounded = np.round(numpy_values[value_name], 4) + 0.0

    # 4자리 반올림 값이 손계산과 같은지 확인한다 (부동소수 오차 1e-9 허용)
    is_matched = bool(np.allclose(numpy_rounded, hand_value, atol=1e-9))

    # 결과를 모은다
    match_results.append(is_matched)

    # 한 줄 출력한다
    print(
        f"{value_name:<11} {str(hand_value):<28} "
        f"{str(numpy_rounded.tolist()):<28} {is_matched}"
    )

# 전체 결과를 출력한다
print(f"\n손계산과 NumPy 4자리 일치: {sum(match_results)}/{len(match_results)}개")

항목          손계산                          NumPy (4자리 반올림)              일치
z1          [0.1, 0.3]                   [0.1, 0.3]                   True
a1          [0.525, 0.5744]              [0.525, 0.5744]              True
z2          0.6072                       0.6072                       True
y_pred      0.6473                       0.6473                       True
dL/dy_pred  -1.5449                      -1.5449                      True
dL/dz2      -0.3527                      -0.3527                      True
dL/dW2      [-0.1852, -0.2026]           [-0.1852, -0.2026]           True
dL/da1      [-0.1764, -0.2116]           [-0.1764, -0.2116]           True
dL/dz1      [-0.044, -0.0517]            [-0.044, -0.0517]            True
dL/dW1      [[-0.044, 0.0], [-0.0517, 0.0]] [[-0.044, 0.0], [-0.0517, 0.0]] True

손계산과 NumPy 4자리 일치: 10/10개


## 6. 수치 미분으로 한 번 더 검증 (Gradient Check)

연쇄 법칙이 맞다면, $W_1$의 각 원소를 $h$만큼 앞뒤로 바꿔 손실 변화를 재는 중심차분 결과와도 같아야 한다.

$$
\frac{\partial L}{\partial W_{1,ij}} \approx \frac{L(W_{1,ij} + h) - L(W_{1,ij} - h)}{2h}, \quad h = 10^{-5}
$$

In [5]:
# 중심차분으로 ∂L/∂W1을 구한다 (src/calculus.py의 numerical_gradient 사용)
numerical_hidden_gradient = numerical_hidden_weight_gradient(
    example.input_vector, example.parameters, example.target
)

# 역전파 결과와 수치 미분 결과의 최대 차이
max_difference = float(
    np.max(np.abs(numerical_hidden_gradient - gradients.hidden_weights))
)

# 역전파 결과를 출력한다
print("역전파  dL/dW1 =", (np.round(gradients.hidden_weights, 6) + 0.0).tolist())

# 수치 미분 결과를 출력한다
print("수치미분 dL/dW1 =", (np.round(numerical_hidden_gradient, 6) + 0.0).tolist())

# 최대 차이를 출력한다
print(f"최대 차이 = {max_difference:.2e}")

역전파  dL/dW1 = [[-0.043979, 0.0], [-0.051734, 0.0]]
수치미분 dL/dW1 = [[-0.043979, 0.0], [-0.051734, 0.0]]
최대 차이 = 1.06e-11


## 정리

- 순전파는 $x \rightarrow z_1 \rightarrow a_1 \rightarrow z_2 \rightarrow \hat{y} \rightarrow L$ 순서로 계산하고, 역전파는 같은 사슬을 거꾸로 따라가며 국소 미분을 곱한다.
- 체크포인트 두 곳
    - $\partial L/\partial z_2 = \hat{y} - y = -0.3527$ (Sigmoid + BCE의 간단한 형태)
    - $\partial L/\partial z_1 = [-0.0440, -0.0517]$ (원소별 곱 $\odot\,\sigma'(z_1)$)
- 손계산, NumPy 코드, 수치 미분 세 방법이 같은 값을 냈다.
- 이 과정이 PyTorch 같은 자동 미분(AutoGrad) 엔진이 내부에서 하는 일이다. 연산마다 국소 미분을 알고 있으면 사슬을 따라 곱하기만 하면 된다.